# Fine-tuning de modelo para sumarização de diálogos de atendimento ao cliente pelo Twitter

- Aplica modelos de sumarização (sem fine-tuning) aos diálogos
  
- Avalia a performance dos modelos com a métrica ROUGE

## Configurações iniciais

In [ ]:
import os

import tqdm
from typing import List

from datasets import Dataset
import evaluate
from rich import print
import pandas as pd
from langchain_community.llms import HuggingFacePipeline
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
import torch

In [3]:
LLM_MODEL_PARAMS = {
    't5_small': {
        'model_name': 'google-t5/t5-small',
        'auto_model_class': AutoModelForSeq2SeqLM,
        'task_pipeline': 'text-generation'
    },
    'falconsai': {
        'model_name': 'Falconsai/text_summarization',
        'auto_model_class': AutoModelForSeq2SeqLM,
        'task_pipeline': 'text-generation'
    }
}

# =============================
MODEL_NAME = 'falconsai'
# MODEL_NAME = 't5_small'


N_MAX_SAMPLES_EVAL = 2327
# =============================


LLM_MODEL_NAME = LLM_MODEL_PARAMS[MODEL_NAME]['model_name']
TASK_PIPLINE = LLM_MODEL_PARAMS[MODEL_NAME]['task_pipeline']
AUTO_MODEL_CLASS = LLM_MODEL_PARAMS[MODEL_NAME]['auto_model_class']

DEVICE = 'cuda'

PRED_COL_NAME = 'summary_pred'

PATH_PREPARED_DATASET_TRAIN = f'../data/interim/summarization_train.csv'
PATH_PREPARED_DATASET_VALID = f'../data/interim/summarization_valid.csv'
PATH_PREPARED_DATASET_TEST = f'../data/interim/summarization_test.csv'

MODEL_PATH = r'/home/msc/Downloads/hf_models'

In [4]:
df_tweets_summ = pd.read_csv(PATH_PREPARED_DATASET_TRAIN)
print(df_tweets_summ.shape)
df_tweets_summ.head()

(866, 6)

,conversation_id,tweet_ids,created_at_list,tweet_texts,human_summary,elapsed_time
0,b065262210783596c1fe79466b8f8985,"[87068, 87069, 87072, 87076]","{87068: Timestamp('2017-11-30 14:29:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Health and activity functions are not working ...,1 days 17:43:08
1,1e1d8fd4f95c984fb78687c9e946dc97,"[607293, 607296, 607297, 607297]","{607293: Timestamp('2017-11-22 12:16:27+0000',...",@115850 hi team! i m planning to get Apple Air...,Customer is eager to know about the replacemen...,0 days 00:56:46
2,b5773077fa55c381260390472deff4c2,"[428555, 428556, 739293, 739295]","{428555: Timestamp('2017-10-10 13:33:05+0000',...",@AskAmex Where do I write to address a custome...,Customer having a query about delta card. Agen...,8 days 03:05:49
3,3574d6b4418cb546a90d1561bacd66a2,"[587935, 587938, 587939, 587942, 587942]","{587935: Timestamp('2017-12-03 15:55:31+0000',...","@AmazonHelp @115821 Wow, expected 4 packages y...",The customer have a problem. The agent is very...,0 days 01:02:56
4,f30d2bbc15e7ff32244761b38b67d160,"[861405, 861406]","{861405: Timestamp('2017-10-13 14:20:19+0000',...",@GWRHelp App. I havent been able to purchase a...,Customer cannot purchase a train ticket on the...,0 days 00:18:07


### Checa amostras

In [5]:
idx_test = 1
print(f'Dialógo Twitter:\n{df_tweets_summ.iloc[idx_test].tweet_texts}')
print(f'Sumarização:\n{df_tweets_summ.iloc[idx_test].human_summary}')

Dialógo Twitter:
@115850 hi team! i m planning to get Apple AirPods ! it shows on the website it has 10 days replacement warranty, 
can u explain me what is it ? @264322 We've a 10days replacement policy if the item you received is damaged or 
defective. ^SH @264322 Yes, headsets/ earphones are not eligible for remorse returns. In case of any damage/ defect
you can reach out to us, we'll check and help you accordingly. ^VN

Sumarização:
Customer is eager to know about the replacement policy on the earphones he wishes to buy. Agent stated that it only
applies if the received item is defective or damaged.

### Define modelo LLM

In [6]:
local_model_path = f'{MODEL_PATH}/{LLM_MODEL_NAME}'

tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AUTO_MODEL_CLASS.from_pretrained(
    local_model_path,
    torch_dtype=torch.float32,
    device_map="auto"
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [7]:
df_eval = df_tweets_summ.head(N_MAX_SAMPLES_EVAL)
df_eval.rename(columns={'tweet_texts': 'inputs'}, inplace=True)
print(df_eval.shape)
df_eval.head(1)

(866, 6)

,conversation_id,tweet_ids,created_at_list,inputs,human_summary,elapsed_time
0,b065262210783596c1fe79466b8f8985,"[87068, 87069, 87072, 87076]","{87068: Timestamp('2017-11-30 14:29:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Health and activity functions are not working ...,1 days 17:43:08


In [ ]:
def llm_summarization(input_text: str, model=model, tokenizer=tokenizer) -> str:
    prompt = f'summarize: {input_text}'

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        inputs['input_ids'], 
        max_new_tokens=100,
        do_sample=False,
        
        max_length=700, 
        min_length=40,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
def run_llm_summarization(
    df: pd.DataFrame,
    llm_tokenizer=tokenizer,
    llm_model: HuggingFacePipeline = None,
) -> List[str]:
    res = []
    for _, df_i in tqdm(df.iterrows(), total=len(df)):
        res_llm = llm_summarization(
            input_text=df_i.inputs
        )
        res.append(res_llm)
    return res

In [10]:
res_model = run_llm_summarization(
    df=df_eval.head(N_MAX_SAMPLES_EVAL),
)
print(f'🤖 {LLM_MODEL_NAME}:\n\n{res_model[:3]} ...\n')

100%|██████████| 866/866 [07:59<00:00,  1.81it/s]


🤖 Falconsai/text_summarization:

['So neither my iPhone nor my Apple Watch are recording my steps/activity, and Health doesn’t recognise either 
source anymore for some reason. @135060 Let’s move to DM and look into this a bit more. When reaching out in DM, 
let us know when this started happening please.', '@264322 Yes, headsets/ earphones are not eligible for remorse 
returns. SH @264322 Yes, headsets/ earphones are not eligible for remorse returns.', "@AskAmex Signed up for new 
card with Delta to book immediately book tix. Card number didn't come up. Customer svce refused to help. @216929 
Good morning, thanks for reaching out."] ...

In [11]:
df_eval[PRED_COL_NAME] = res_model

In [12]:
model_name_output = LLM_MODEL_NAME.split('/')[0].lower()
output_path = f'../data/processed/df_eval_{model_name_output}.csv'

df_eval.to_csv(output_path, index=False)

print(f'CSV exportado: {output_path}')

CSV exportado: ../data/processed/df_eval_falconsai.csv

### Checa sumarização

In [13]:
n_test = 7

print(df_eval[['inputs', PRED_COL_NAME]].iloc[n_test].to_dict())

{
    'inputs': '@USCellularCares when is thrtr going to be an android update last one was April? @213129 The current
android version for the Galaxy S7 is 7.0.0 and the base band version is G930R4TYU4BQE1. Is that what you have? ^KJ 
@USCellularCares Yes but why is it still on April when there is a September from samsung @213129 I have contacted 
Samsung and the current version is the most recent available for this phone at this time. ^KJ @213129 Bob, once 
Samsung shares the updates with us, we will then share them with customers. ^AW',
    'summary_pred': '@USCellularCares The current android version for the Galaxy S7 is 7.0.0 and the base band 
version is G930R4TYU4BQE1. @213129 I have contacted Samsung and the current version is the most recent available 
for this phone at this time.'
}

## Calcula métricas

In [14]:
print(df_eval.shape)
df_eval.head(1)

(866, 7)

,conversation_id,tweet_ids,created_at_list,inputs,human_summary,elapsed_time,summary_pred
0,b065262210783596c1fe79466b8f8985,"[87068, 87069, 87072, 87076]","{87068: Timestamp('2017-11-30 14:29:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Health and activity functions are not working ...,1 days 17:43:08,So neither my iPhone nor my Apple Watch are re...


### Métrica [ROUGE (Recall-Oriented Understudy for Gisting Evaluation)](https://en.wikipedia.org/wiki/ROUGE_(metric))

In [15]:
def calc_rouge_metric(
    df: pd.DataFrame,
    target_name: str = 'target',
    pred_name: str = 'pred'
) -> pd.DataFrame:
    rouge_metric = evaluate.load('rouge')

    df_hf = Dataset.from_pandas(df)

    score = rouge_metric.compute(
        predictions=df_hf[target_name],
        references=df_hf[pred_name]
    )

    df_metrics = pd.DataFrame({LLM_MODEL_NAME: score})
    return df_metrics

In [16]:
df_metrics = calc_rouge_metric(df=df_eval, target_name='human_summary', pred_name=PRED_COL_NAME)
df_metrics = df_metrics.T.reset_index().rename(columns={'index': 'model'})
df_metrics

,model,rouge1,rouge2,rougeL,rougeLsum
0,Falconsai/text_summarization,0.285095,0.125963,0.233731,0.233686


### Exporta métricas em CSV

In [18]:
model_name_output = LLM_MODEL_NAME.split('/')[0].lower().replace('-', '_')
output_path = f'../data/processed/df_metrics_{model_name_output}.csv'

df_metrics.to_csv(output_path, index=False)

print(f'CSV exportado: {output_path}')

CSV exportado: ../data/processed/df_metrics_falconsai.csv

In [19]:
# teste
df_metrics_test = pd.read_csv(output_path)
df_metrics_test.head()

,model,rouge1,rouge2,rougeL,rougeLsum
0,Falconsai/text_summarization,0.285095,0.125963,0.233731,0.233686
